In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
from src.abg_logic import interpret_abg

print(interpret_abg(7.2, 60, 24))

{'primary': 'Respiratory Acidosis'}


In [3]:
import pandas as pd
import numpy as np

data = []

for _ in range(2000):
    ph = np.random.uniform(7.1, 7.6)
    pco2 = np.random.uniform(20, 80)
    hco3 = np.random.uniform(10, 40)

    label = interpret_abg(ph, pco2, hco3)["primary"]

    data.append([ph, pco2, hco3, label])

df = pd.DataFrame(data, columns=["ph", "pco2", "hco3", "label"])

df.head()

,ph,pco2,hco3,label
0,7.345503,62.053362,31.895505,Respiratory Acidosis
1,7.363008,20.759537,17.577560,Mixed/Compensated
2,7.383660,22.410493,21.556595,Mixed/Compensated
3,7.457892,57.579466,17.539349,Mixed/Compensated
4,7.385606,32.766539,30.558307,Mixed/Compensated


In [4]:
df.to_csv("../data/abg_dataset.csv", index=False)

In [5]:
import pandas as pd
import numpy as np

def interpret_abg(ph, pco2, hco3):
    if ph < 7.35 and pco2 > 45:
        return "Respiratory Acidosis"
    elif ph < 7.35 and hco3 < 22:
        return "Metabolic Acidosis"
    elif ph > 7.45 and pco2 < 35:
        return "Respiratory Alkalosis"
    elif ph > 7.45 and hco3 > 26:
        return "Metabolic Alkalosis"
    else:
        return "Mixed/Compensated"

data = []

for _ in range(3000):
    ph = np.random.uniform(7.1, 7.6)
    pco2 = np.random.uniform(20, 80)
    hco3 = np.random.uniform(10, 40)
    po2 = np.random.uniform(50, 120)

    # Ventilator settings (simulate realistic ranges)
    rr = np.random.randint(8, 30)         # Respiratory rate
    tv = np.random.uniform(300, 700)      # Tidal volume (ml)
    fio2 = np.random.uniform(0.21, 1.0)   # Oxygen %
    peep = np.random.uniform(5, 15)       # PEEP

    label = interpret_abg(ph, pco2, hco3)

    data.append([ph, pco2, hco3, po2, rr, tv, fio2, peep, label])

df = pd.DataFrame(data, columns=[
    "ph", "pco2", "hco3", "po2",
    "rr", "tv", "fio2", "peep",
    "label"
])

df.head()

,ph,pco2,hco3,po2,rr,tv,fio2,peep,label
0,7.147728,65.887858,16.553192,101.466543,12,467.506742,0.318811,11.049509,Respiratory Acidosis
1,7.168081,26.171735,28.424259,69.200334,22,545.087778,0.776161,6.192262,Mixed/Compensated
2,7.595524,25.604441,38.277518,75.673307,13,574.062395,0.696736,7.315594,Respiratory Alkalosis
3,7.279413,70.688206,21.359217,119.398126,26,487.530873,0.401061,12.937056,Respiratory Acidosis
4,7.106573,31.625941,17.776722,70.524611,19,615.622559,0.497539,10.384427,Metabolic Acidosis


In [6]:
def ventilator_label(ph, pco2, po2):
    if pco2 > 45:
        return "Increase Ventilation"
    elif pco2 < 35:
        return "Decrease Ventilation"
    elif po2 < 80:
        return "Improve Oxygenation"
    else:
        return "Stable"

df["vent_label"] = df.apply(
    lambda x: ventilator_label(x["ph"], x["pco2"], x["po2"]),
    axis=1
)

In [7]:
from sklearn.model_selection import train_test_split

X = df[["ph", "pco2", "hco3", "po2", "rr", "tv", "fio2", "peep"]]
y = df["vent_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

model = RandomForestClassifier()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))

                      precision    recall  f1-score   support

Decrease Ventilation       1.00      1.00      1.00       137
 Improve Oxygenation       1.00      1.00      1.00        49
Increase Ventilation       1.00      1.00      1.00       353
              Stable       1.00      1.00      1.00        61

            accuracy                           1.00       600
           macro avg       1.00      1.00      1.00       600
        weighted avg       1.00      1.00      1.00       600



In [9]:
sample = [[7.25, 60, 24, 70, 18, 500, 0.5, 8]]
model.predict(sample)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


array(['Increase Ventilation'], dtype=object)

In [10]:
import pickle

with open("../model.pkl", "wb") as f:
    pickle.dump(model, f)